# Notebook 3: Analyze Visium H&E Data

**Source:** [Squidpy Tutorial — Visium H&E](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_visium_hne.html)

This notebook demonstrates the full Squidpy spatial analysis workflow on **Hematoxylin & Eosin (H&E)** stained Visium data from a mouse brain coronal section.

## Topics Covered
1. Loading pre-processed Visium H&E data
2. Multi-scale image feature extraction from H&E image
3. Image-based clustering vs. gene-based clustering
4. Spatial neighborhood graph construction
5. Neighborhood enrichment analysis
6. Co-occurrence scoring across spatial dimensions
7. Ligand-receptor interaction analysis

In [1]:
!pip install scanpy squidpy leidenalg python-igraph seaborn openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of spatialdata to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of spatialdata to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of dask[array,dataframe] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of ome-zarr to determine which version is compatible with other requirements. This could take a 

## 0. Import Libraries

In [2]:
%matplotlib inline

import numpy as np
import pandas as pd

import anndata as ad
import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

squidpy==1.8.1


## 1. Load Data

We load the pre-processed mouse brain Visium H&E dataset provided by Squidpy.
- **Cluster annotations** were derived using the Allen Brain Atlas and Mouse Brain gene expression atlas
- Pre-processing pipeline follows the standard Scanpy workflow (see Notebook 1)

In [3]:
# Load pre-processed Visium H&E dataset
img = sq.datasets.visium_hne_image()
adata = sq.datasets.visium_hne_adata()

print(adata)
print("\nAvailable clusters:", adata.obs["cluster"].cat.categories.tolist())

INFO     Downloading visium_hne_image.tiff from                                                                    
         https://exampledata.scverse.org/squidpy/figshare/visium_hne_image.tiff                                    


  0%|                                               | 0.00/398M [00:00<?, ?B/s]

INFO     Downloading visium_hne_adata.h5ad from                                                                    
         https://exampledata.scverse.org/squidpy/figshare/visium_hne_adata.h5ad                                    


  0%|                                               | 0.00/329M [00:00<?, ?B/s]

AnnData object with n_obs × n_vars = 2688 × 18078
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_counts', 'leiden', 'cluster'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cluster_colors', 'hvg', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'rank_genes_groups', 'spatial', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

Available clusters: ['Cortex_1', 'Cortex_2', 'Cortex_3', 'Cortex_4', 'Cortex_5', 'Fiber_tract', 'Hippocampus', 'Hypot

In [4]:
# Visualize cluster annotation in spatial context
sq.pl.spatial_scatter(adata, color="cluster")

<Figure size 640x480 with 1 Axes>

## 2. Image Feature Extraction from H&E

Even from a standard H&E image, we can extract **summary statistics** per spot at different spatial scales. A larger scale means more surrounding tissue context is included.

We extract summary features at:
- **scale=1.0** — original resolution
- **scale=2.0** — zoomed out (more context)

In [5]:
# Extract summary features at two spatial scales
for scale in [1.0, 2.0]:
    feature_name = f"features_summary_scale{scale}"
    sq.im.calculate_image_features(
        adata,
        img.compute(),
        features="summary",
        key_added=feature_name,
        n_jobs=4,
        scale=scale
    )
    print(f"Extracted features at scale={scale} → stored in adata.obsm['{feature_name}']")

  0%|          | 0/2688 [00:00<?, ?/s]

Extracted features at scale=1.0 → stored in adata.obsm['features_summary_scale1.0']


  0%|          | 0/2688 [00:00<?, ?/s]

Extracted features at scale=2.0 → stored in adata.obsm['features_summary_scale2.0']


In [6]:
# Combine features from both scales into one matrix
adata.obsm["features"] = pd.concat(
    [adata.obsm[f] for f in adata.obsm.keys() if "features_summary" in f],
    axis="columns"
)

# Ensure unique feature column names
adata.obsm["features"].columns = ad.utils.make_index_unique(
    adata.obsm["features"].columns
)

print(f"Combined feature matrix: {adata.obsm['features'].shape[0]} spots × {adata.obsm['features'].shape[1]} features")

Combined feature matrix: 2688 spots × 30 features


## 3. Clustering in Image Feature Space

We cluster spots based on image morphology and compare to gene-expression clusters.

In [7]:
def cluster_features(features: pd.DataFrame, like=None) -> pd.Series:
    """Compute Leiden clustering of image features.

    Parameters
    ----------
    features
        DataFrame of image features (spots × features).
    like
        Substring filter for column names.

    Returns
    -------
    pd.Series of Leiden cluster labels.
    """
    if like is not None:
        features = features.filter(like=like)

    tmp_adata = ad.AnnData(features)
    sc.pp.scale(tmp_adata)   # scale features before PCA
    sc.pp.pca(tmp_adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(tmp_adata)
    sc.tl.leiden(tmp_adata)

    return tmp_adata.obs["leiden"]

In [8]:
# Cluster using image summary features
adata.obs["features_cluster"] = cluster_features(
    adata.obsm["features"], like="summary"
)

# Compare image feature clusters vs. gene-expression clusters side by side
sq.pl.spatial_scatter(adata, color=["features_cluster", "cluster"])

/tmp/ipykernel_9055/1235674224.py:22: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(tmp_adata)


<Figure size 1455.6x480 with 2 Axes>

**Interpretation:**
- The *Fiber_tract* cluster is well-recapitulated by image features (similar structure visible in both)
- Hippocampus regions show rough correspondence between image and gene clusters
- The cortex differs: gene clusters show layered cortical structure, while image clusters show regional differences
- Image and gene features provide **complementary** views of the tissue

## 4. Spatial Neighborhood Graph

All spatial statistics in Squidpy require a **spatial connectivity matrix** — a graph where each spot is connected to its spatial neighbors.

`sq.gr.spatial_neighbors()` builds this graph from `adata.obsm['spatial']`.

In [9]:
# Build spatial connectivity graph
sq.gr.spatial_neighbors(adata)

print("Spatial graph stored in:")
print("  adata.obsp['spatial_connectivities'] — adjacency matrix")
print("  adata.obsp['spatial_distances']      — distance matrix")
print(f"  Graph shape: {adata.obsp['spatial_connectivities'].shape}")

INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
Spatial graph stored in:
  adata.obsp['spatial_connectivities'] — adjacency matrix
  adata.obsp['spatial_distances']      — distance matrix
  Graph shape: (2688, 2688)


## 5. Neighborhood Enrichment Analysis

**Neighborhood enrichment** quantifies whether clusters tend to be spatially co-localized.

- **High score** → two clusters are frequently neighbors (spatially enriched)
- **Low score** → two clusters rarely co-occur in the same neighborhood (spatially depleted)

This is a **permutation-based test** (`n_perms=1000` by default).

In [10]:
# Compute neighborhood enrichment scores
sq.gr.nhood_enrichment(adata, cluster_key="cluster")

# Visualize as a heatmap
sq.pl.nhood_enrichment(
    adata,
    cluster_key="cluster",
    figsize=(8, 8),
    title="Neighborhood Enrichment Score"
)

  0%|          | 0/1000 [00:00<?, ?/s]

<Figure size 800x800 with 4 Axes>

**Key Finding:**
The Hippocampus shows strong internal neighborhood enrichment:
- *Pyramidal_layer_dentate_gyrus* ↔ *Pyramidal_layer* ↔ *Hippocampus* are frequently neighbors
- This reflects the layered anatomical organization of the hippocampal formation

## 6. Co-occurrence Scoring

**Co-occurrence** measures the conditional probability of observing cluster B given cluster A at increasing radii:

$$\text{score} = \frac{p(\text{exp} | \text{cond})}{p(\text{exp})}$$

- Score **> 1** → cluster exp appears more often near cluster cond than expected by chance
- Score **< 1** → cluster exp is depleted around cluster cond
- Computed across increasing radii → shows at what **spatial distance** clusters interact

In [11]:
# Compute co-occurrence scores
sq.gr.co_occurrence(adata, cluster_key="cluster")

# Visualize co-occurrence around the Hippocampus cluster
sq.pl.co_occurrence(
    adata,
    cluster_key="cluster",
    clusters="Hippocampus",
    figsize=(10, 4)
)

<Figure size 1000x400 with 1 Axes>

**Interpretation:**
- *Pyramidal_layer* co-occurs strongly with *Hippocampus* at **short distances**
- This confirms the tight spatial relationship seen in the neighborhood enrichment analysis
- Distance units = pixels in the Visium source image (same units as `adata.obsm['spatial']`)

## 7. Ligand-Receptor Interaction Analysis

After identifying spatially co-localized clusters, we can ask: **what molecular signals might drive communication between them?**

Squidpy re-implements **CellPhoneDB** with the **Omnipath** ligand-receptor database.  
`sq.gr.ligrec()` performs a permutation test across all cluster pairs and all annotated L-R pairs.

Here we focus on the Hippocampus → Pyramidal layer communication axis.

In [12]:
# Run ligand-receptor analysis (100 permutations for speed; use 1000 for publication)
sq.gr.ligrec(
    adata,
    n_perms=100,
    cluster_key="cluster"
)

0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

  0%|          | 0/100 [00:00<?, ?permutation/s]

In [13]:
# Visualize significant interactions
# Source: Hippocampus | Targets: Pyramidal layer clusters
# Filter: mean expression > 3, adjusted p-value < 1e-4
sq.pl.ligrec(
    adata,
    cluster_key="cluster",
    source_groups="Hippocampus",
    target_groups=["Pyramidal_layer", "Pyramidal_layer_dentate_gyrus"],
    means_range=(3, np.inf),
    alpha=1e-4,
    swap_axes=True
)

/usr/local/lib/python3.12/dist-packages/squidpy/pl/_ligrec.py:248: FutureWarning: The method uns_keys is deprecated and will be removed in the future. Use uns instead of uns_keys. (e.g. `k in adata.uns` or `sorted(adata.uns)`)
  if cluster_key not in adata.uns_keys():
/usr/local/lib/python3.12/dist-packages/squidpy/pl/_ligrec.py:36: UserWarning: Over 500 categories found. Plot would be very large.
  super().__init__(*args, **kwargs)


<Figure size 25236x250 with 7 Axes>

**Interpretation of the dotplot:**
- Each **dot** = one ligand-receptor pair
- **Dot size** = significance (–log10 adjusted p-value)
- **Dot color** = mean expression of the interaction pair
- High-scoring pairs are candidate mediators of Hippocampus ↔ Pyramidal layer communication

> These interactions are candidates only — experimental validation would be required to confirm functional relevance.

## Summary

In this notebook we applied the full Squidpy spatial analysis pipeline to H&E Visium data:

| Analysis | Function | What it reveals |
|----------|----------|-----------------|
| Image feature extraction | `sq.im.calculate_image_features()` | Tissue morphology per spot |
| Image clustering | Leiden on feature PCA | Morphology-based regions |
| Spatial graph | `sq.gr.spatial_neighbors()` | Connectivity structure |
| Neighborhood enrichment | `sq.gr.nhood_enrichment()` | Which clusters co-localize |
| Co-occurrence | `sq.gr.co_occurrence()` | At what distance clusters interact |
| Ligand-receptor | `sq.gr.ligrec()` | Candidate cell-cell communication signals |
